In [ ]:
# autoreload
%load_ext autoreload
%autoreload 2
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
import gc
import numpy as np
import gradio as gr
import json 
import re
import subprocess
import IPython.display as ipd
import torch
import torchaudio

from einops import rearrange
from safetensors.torch import load_file
from torch.nn import functional as F
from torchaudio import transforms as T

from stable_audio_tools.interface.aeiou import audio_spectrogram_image
from stable_audio_tools.inference.generation import generate_diffusion_cond, generate_diffusion_cond_inpaint, generate_diffusion_uncond
from stable_audio_tools.models.factory import create_model_from_config
from stable_audio_tools.models.pretrained import get_pretrained_model
from stable_audio_tools.models.utils import copy_state_dict, load_ckpt_state_dict
from stable_audio_tools.inference.utils import prepare_audio
from stable_audio_tools.loraw.network import LoRAMerger, create_lora_from_config

from stable_audio_tools.interface.interfaces.diffusion_cond import create_diffusion_cond_ui

model = None
model_type = None
sample_rate = 32000
sample_size = 1920000

def load_model(model_config=None, model_ckpt_path=None, pretrained_name=None, pretransform_ckpt_path=None, device="cuda", model_half=False):
    global model, sample_rate, sample_size
    
    if pretrained_name is not None:
        print(f"Loading pretrained model {pretrained_name}")
        model, model_config = get_pretrained_model(pretrained_name)

    elif model_config is not None and model_ckpt_path is not None:
        print(f"Creating model from config")
        model = create_model_from_config(model_config)

        print(f"Loading model checkpoint from {model_ckpt_path}")
        # Load checkpoint
        copy_state_dict(model, load_ckpt_state_dict(model_ckpt_path))
        #model.load_state_dict(load_ckpt_state_dict(model_ckpt_path))

    sample_rate = model_config["sample_rate"]
    sample_size = model_config["sample_size"]

    if pretransform_ckpt_path is not None:
        print(f"Loading pretransform checkpoint from {pretransform_ckpt_path}")
        model.pretransform.load_state_dict(load_ckpt_state_dict(pretransform_ckpt_path), strict=False)
        print(f"Done loading pretransform")

    model.to(device).eval().requires_grad_(False)

    if model_half:
        model.to(torch.float16)
    
    lora = create_lora_from_config(model_config, model)
    return lora, model, model_config


model_config_path="/home/zachary/code/stable-audio-tools/stable_audio_tools/configs/model_configs/txt2audio/sao_short_inpaint.json"
ckpt_path="/home/zachary/.cache/huggingface/hub/models--stabilityai--stable-audio-open-1.0/snapshots/f21265c1e2710b3bd2386596943f0007f55f802e/model.safetensors"

if model_config_path is not None:
        # Load config from json file
    with open(model_config_path) as f:
        model_config = json.load(f)
else:
    model_config = None

device = "cuda" if torch.cuda.is_available() else "cpu"

lora, sao, conf = load_model(model_config, ckpt_path,  device=device)

In [ ]:
import torch

lora_state_d = torch.load("/home/zachary/checkpoints/s2s/ossl2_experiments/sok583sy/checkpoints/epoch=36-step=80000.ckpt", map_location="cpu")
lora.load_weights(lora_state_d)
lora.activate()

In [ ]:
ref_audio, sr = torchaudio.load("/home/zachary/code/stable-audio-tools/notebooks/demo_cfg_4_320962_9da78d7d47814a3c52fa.wav")
ref_audio = ref_audio[:, 524288:524288*2]

In [ ]:
# encode reference audio
ref_audio_prepared = prepare_audio(ref_audio, sr, sample_rate, sample_size, 2, device=device)
with torch.no_grad():
    ref_latents = sao.pretransform.encode(ref_audio_prepared)

In [ ]:
sao = sao.to(device)
sao.model.model = torch.compile(sao.model.model)

In [ ]:
from stable_audio_tools.inference.generation import generate_diffusion_cond_blockar
from stable_audio_tools.models.inpainting import random_inpaint_mask

sample_rate = model_config["sample_rate"]
sample_size = model_config["sample_size"]




# Set up text and timing conditioning
conditioning = [{
    "prompt": "this song is a chill, 110 bpm lo-fi hip hop beat with smooth drums and jazzy instrumentation. it would be perfect for studying, relaxing, or just vibing out.",
    "seconds_start": 0, 
    "seconds_total": 12
}]

# mask latents
inpaint_masked_input, inpaint_mask = random_inpaint_mask(ref_latents, torch.ones_like(ref_latents), **model_config['training']['inpainting']['mask_kwargs'])


conditioning_tensors = model.conditioner(conditioning, device)
conditioning_tensors['inpaint_mask'] = [inpaint_mask]
conditioning_tensors['inpaint_masked_input'] = [inpaint_masked_input]

output = generate_diffusion_cond_blockar(
    sao,
    steps=50,
    cfg_scale=7,
    conditioning_tensors=conditioning_tensors,s
    sample_size=sample_size,
    # init_audio=(sr, ref_audio), # turn this on if you want ~10 seconds of initial audio to condition on
    sigma_min=0,
    sigma_max=1,
    sampler_type="v-ddim",
    device=device,
    ar_style='outpaint',
    block_size=98304,
    generation_length=983040
)

output = rearrange(output, "b d n -> d (b n)")
ipd.Audio(output.cpu().numpy(), rate=sample_rate)